In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

In [8]:


# add whatever and however much equities you would like
tickers = input("Enter tickers, no need for commas just a space after every entry").split()

data = []

for t in tickers:
    try:
        stock = yf.Ticker(t)
        info = stock.info
        quarterly = stock.quarterly_income_stmt.T
        
        if info is None or len(info) == 0:
            continue
        
        eps = ["Diluted EPS", "Basic EPS"] 
        eps_growth = None
        
        for i in eps:
            if i in quarterly.columns:
                s = quarterly[i].dropna() # removes NaNs
                s = s.sort_index() # chronological order
                if len(s) > 1:
                    eps_growth = s.pct_change() 
                    eps_growth = eps_growth.iloc[-1] * 100 # need to use iloc here because quarterly returns a list of all four quarters
                break

        
        total_debt = info.get("totalDebt", 0) # sets default to 0 if no key for debt or cash 
        cash = info.get("totalCash", 0)
        mktcap = info.get("marketCap")
        fcf = info.get("freeCashflow")
        
        if fcf is not None and mktcap is not None:
            fcf_yield = fcf/mktcap
        else:
            fcf_yield = None

        ebitda = info.get("ebitda")

        if ebitda is not None and ebitda != 0:
            nb_ebitda = (total_debt - cash) / ebitda
        else:
            nb_ebitda = None
            
        data.append({
            "ticker": t, 
            "pe": info.get("trailingPE"), # trailing PE compares current share price to the total EPS 
            "ev_ebitda": info.get("enterpriseToEbitda"),
            "oper_margin": info.get("operatingMargins"),
            "roe": info.get("returnOnEquity"),
            "net_debt": total_debt - cash,
            "fcf_yield": fcf_yield,
            "ebitda": info.get("ebitda"),
            "beta": info.get("beta"),
            "eps_growth": eps_growth,
            "netdebt_ebitda": nb_ebitda,
            "sector": info.get("sector")
        })
        
# handling missing information case
 
    except Exception as e: 
        print(f"Skipping {t}: {e}")


# setting up table

df = pd.DataFrame(data) # converts list of dictionaries to the appropriate table
df = df.set_index("ticker") # sets the respective ticker as the index
df = df.replace([np.inf, -np.inf], np.nan) # sets any infinite value to NaN
df = df.dropna(thresh=5) # keeps rows with at least 5 non-NaN 

score_df = pd.DataFrame(index=df.index)

# columns for the above table
higher = ["fcf_yield","oper_margin","roe"] # the higher the metric the better 
lower = ["pe","ev_ebitda","beta"] # the lower the metric the better

for metric in higher:
    score_df[metric] = df.groupby("sector")[metric].transform(  # transform function applies the lambda function to each company while keeping the same distribution shape
        lambda z: (z - z.mean()) / (z.std(ddof=0) + 1e-10)   # z-score function
    )

for metric in lower:
    score_df[metric] = -df.groupby("sector")[metric].transform( # negate to follow direction change of metrics 
        lambda z: (z - z.mean()) / (z.std(ddof=0) + 1e-10) 
    )
# by using the transform function we are able to effectively apply a z-score rating for each metric for each company, where it can then be summed to create a final score, where the greater the better of a long position you are in
score_df["long_score"] = score_df.sum(axis=1) # represents the long score for each respective metric seen in the data table

screen = df[                               # only lets companies that satisfy these Boolean statements in
    (df["ev_ebitda"] > 0) &
    (df["ebitda"] > 0) &
    (df["net_debt"] / df["ebitda"] < 3) 
]

final = score_df.loc[screen.index].sort_values("long_score", ascending=False) # sorts the values from highest to lowest 
finalists = final.head(10).index.tolist()
final.head(10) #only takes the top ten


Enter tickers, no need for commas just a space after every entry A AAPL ABBV ABC ABMD ABT ACN ADBE ADI ADM ADP ADSK AEE AEP AES AFL AIG AIZ AJG AKAM ALB ALGN ALK ALL ALLE AMAT AMCR AMD AME AMGN AMP AMT AMZN ANET ANSS AON AOS APA APD APH APTV ARE ATO ATVI AVB AVGO AVY AWK AXP AZO BA BAC BALL BAX BBWI BBY BDX BEN BF-B BIIB BIO BK BKNG BKR BLK BLL BMY BR BRK-B BSX BWA BXP C CAG CAH CARR CAT CB CBOE CBRE CCI CCL CDAY CDNS CDW CE CEG CF CFG CHD CHRW CHTR CI CINF CL CLX CMA CMCSA CME CMG CMI CMS CNC CNP COF COO COP COST CPRT CPT CRL CRM CSCO CSX CTAS CTLT CTSH CTVA CVS CVX CZR D DAL DD DE DFS DG DGX DHI DHR DIS DISCA DISCK DISH DLR DLTR DOV DOW DPZ DRE DRI DTE DUK DVA DVN DXC DXCM EA EBAY ECL ED EFX EIX EL EMN EMR ENPH EOG EPAM EQIX EQR ERE ESS ETN ETR ETY EVRG EW EXC EXPD EXPE EXR F FANG FAST FB FCX FDX FE FFIV FIS FISV FITB FLT FMC FOX FOXA FRC FRT FSLR FTNT FTV GD GE GPC GPN GPS GRMN GS GWW HAL HAS HBAN HCA HD HES HIG HII HLT HOLX HON HPC HPQ HRL HSIC HST HSY HUM HWM IBM ICE IDXX IEX IFF 

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ABC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BLL"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GPS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INEC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol

,fcf_yield,oper_margin,roe,pe,ev_ebitda,beta,long_score
ticker,,,,,,,
APA,2.989328,1.905257,1.778637,0.581392,1.822078,0.639358,9.716051
MAS,0.280044,-0.002770,7.613647,0.879966,0.381383,-0.813100,8.339170
NEM,1.040459,2.711424,0.690268,0.727193,1.745800,1.141395,8.056539
HAS,0.379047,0.692463,3.600094,NaN,0.299486,1.485655,6.456745
DXC,6.670449,-1.664892,-0.267373,-0.925919,1.109723,1.042099,5.964087
MO,0.397357,3.511081,NaN,1.096908,0.872537,-0.049202,5.828681
ALL,1.102318,-0.677763,0.708622,1.297588,1.307911,2.069922,5.808599
CF,0.328404,0.959671,0.748132,0.751401,1.542178,1.390666,5.720451
MA,-0.364569,1.628497,6.626239,-1.449893,-1.707171,0.466060,5.199164


In [10]:
#----------------------------------------------------------------------------------------
# Further evaluation on the finalist companies
#----------------------------------------------------------------------------------------

data = []  # this will store each row

def wacc(ticker_symbol):
    t = yf.Ticker(ticker_symbol)
    info = t.info

    market_cap = info.get('marketCap', 0)
    total_debt = info.get('totalDebt', 0)

    # tax rate
    try:
        income_stmt = t.income_stmt
        latest_income = income_stmt.iloc[:, 0]

        tax_provision = latest_income.get('TaxProvision')
        pretax_income = latest_income.get('PretaxIncome')

        if pretax_income and pretax_income != 0:
            tax_rate = tax_provision / pretax_income
        else:
            tax_rate = 0.25
    except:
        tax_rate = 0.25

    # CAPM 
    beta = info.get('beta', 1)
    risk_free_rate = 0.0416
    market_return = 0.1639

    cost_of_equity = risk_free_rate + beta * (market_return - risk_free_rate)

    # cost of debt
    try:
        interest_expense = latest_income.get('InterestExpense')
        if interest_expense and total_debt != 0:
            cost_of_debt = abs(interest_expense) / total_debt
        else:
            cost_of_debt = 0.06
    except:
        cost_of_debt = 0.06

    # WACC
    total_value = market_cap + total_debt
    if total_value == 0:
        return np.nan

    wacc = (
        market_cap / total_value * cost_of_equity +
        total_debt / total_value * cost_of_debt * (1 - tax_rate)
    )

    return wacc * 100  

#financial statements

for ticker in finalists:
    
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        w = wacc(ticker)
        

        # valuation metrics
        pe = info.get("trailingPE")
        eps_growth = info.get("earningsGrowth")

        peg = pe / eps_growth if pe and eps_growth else np.nan

        ebitda = info.get("ebitda")
        total_debt_info = info.get("totalDebt", 0)
        total_cash_info = info.get("totalCash", 0)

        net_debt = total_debt_info - total_cash_info
        leverage = net_debt / ebitda if ebitda else np.nan      

        # CAGR
        history = yf.download(ticker, period='5y', interval='3mo', progress=False)
        bv = history['Close'].iloc[0]
        ev = history['Close'].iloc[-1]
        cagr = (ev / bv) ** (1 / 5) - 1

        data.append({
            'Ticker': ticker,
            'WACC': w,
            'PEG': peg,
            'Leverage': leverage,
            '5Y CAGR': cagr
        })

    except Exception as e:
        print(f"Skipping {ticker}: {e}")
df = pd.DataFrame(data)
df


,Ticker,WACC,PEG,Leverage,5Y CAGR
0,APA,7.380233,26.140528,0.814239,Ticker APA 0.142516 dtype: float64
1,MAS,17.181165,96.658410,2.012431,Ticker MAS 0.089788 dtype: float64
2,NEM,9.768334,15.722616,-0.198727,Ticker NEM 0.149775 dtype: float64
3,HAS,8.839689,NaN,1.667825,Ticker HAS 0.039728 dtype: float64
4,DXC,7.050989,NaN,1.727461,Ticker DXC -0.225524 dtype: float64
5,MO,9.239227,14.248484,1.334621,Ticker MO 0.181618 dtype: float64
6,ALL,6.014888,1.644022,0.140836,Ticker ALL 0.172349 dtype: float64
7,CF,8.235666,9.927129,0.459121,Ticker CF 0.202115 dtype: float64
8,MA,12.743172,145.216401,0.502456,Ticker MA 0.095247 dtype: float64
9,BMY,6.344682,185.239315,1.894329,Ticker BMY 0.045925 dtype: float64
